# Q6: Inoculation Sample Complexity for Typosquat Subspace Suppression
Measures how many (typo → correct) fine‑tuning examples are needed to collapse the typosquatting activation subspace.

In [ ]:
!pip install -q torch transformers accelerate peft datasets scikit-learn matplotlib seaborn

In [ ]:
import json, random, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from typing import List, Tuple, Optional
from dataclasses import dataclass
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
@dataclass
class Q6Config:
    DATASET_PATH: str = "typosquat_tool_calls.jsonl"
    FILTER_ADVERSARIAL: bool = True
    MODEL_NAME: str = "Qwen/Qwen2.5-1.5B-Instruct"
    EXTRACTION_LAYERS: Tuple[int, int] = (12, 16)
    MAX_LENGTH: int = 512
    PROBE_TEST_RATIO: float = 0.3
    PROBE_MAX_ITER: int = 1000
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05
    TRAIN_EPOCHS: int = 1
    TRAIN_BATCH_SIZE: int = 4
    TRAIN_LR: float = 2e-4
    SAMPLE_SIZES: List[int] = None
    OUTPUT_DIR: str = "q6_results"
    def __post_init__(self):
        if self.SAMPLE_SIZES is None:
            self.SAMPLE_SIZES = [10, 50, 100, 500]

cfg = Q6Config()
Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# Upload dataset (ZIP or JSONL)
from google.colab import files
import zipfile, io
uploaded = files.upload()
fname = list(uploaded.keys())[0]
if fname.endswith('.zip'):
    with zipfile.ZipFile(io.BytesIO(uploaded[fname])) as zf:
        jsonl = [n for n in zf.namelist() if n.endswith('.jsonl')][0]
        zf.extract(jsonl)
        cfg.DATASET_PATH = jsonl
else:
    cfg.DATASET_PATH = fname
print(f"Using dataset: {cfg.DATASET_PATH}")

In [ ]:
def load_dataset(path, cfg):
    entries = [json.loads(line) for line in open(path)]
    df = pd.DataFrame(entries)
    if cfg.FILTER_ADVERSARIAL:
        df = df[df['is_adversarial'] == True].copy()
    print(f"Filtered to {len(df)} adversarial entries")
    return df

df = load_dataset(cfg.DATASET_PATH, cfg)

In [ ]:
def find_package_span(command, pkg):
    m = re.search(r'\b' + re.escape(pkg) + r'\b', command)
    return (m.start(), m.end()) if m else None

def char_to_token_span(tokenizer, text, cs, ce):
    enc = tokenizer.encode_plus(text, return_offsets_mapping=True)
    offs = enc['offset_mapping']
    ts = te = None
    for i, (s, e) in enumerate(offs):
        if s <= cs < e or s < ce <= e:
            if ts is None: ts = i
            te = i
    return (ts or len(offs)-1, te or len(offs)-1)

def extract_hidden_states(model, tokenizer, commands, packages, layers=(12,16), batch_size=8):
    model.eval()
    all_states = []
    for i in range(0, len(commands), batch_size):
        batch_cmds = commands[i:i+batch_size]
        batch_pkgs = packages[i:i+batch_size]
        inputs = tokenizer(batch_cmds, return_tensors="pt", padding=True, truncation=True, max_length=cfg.MAX_LENGTH)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states
        for j, (cmd, pkg) in enumerate(zip(batch_cmds, batch_pkgs)):
            span = find_package_span(cmd, pkg)
            if span is None:
                states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            else:
                ts, te = char_to_token_span(tokenizer, cmd, *span)
                states = [hidden[l][j, ts:te+1, :].mean(dim=0).float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            if states:
                all_states.append(np.concatenate(states))
    return np.array(all_states)

In [ ]:
def prepare_probe_data(df, rng):
    texts, pkgs, labels = [], [], []
    for _, row in df.iterrows():
        texts.append(row['clean_command']); pkgs.append(row['package_name']); labels.append(0)
        texts.append(row['typo_command']); pkgs.append(row['typo_package']); labels.append(1)
    combined = list(zip(texts, pkgs, labels))
    rng.shuffle(combined)
    texts, pkgs, labels = zip(*combined)
    return list(texts), list(pkgs), list(labels)

def train_probe(X, y, cfg):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=cfg.PROBE_TEST_RATIO, stratify=y, random_state=SEED)
    probe = LogisticRegression(max_iter=cfg.PROBE_MAX_ITER, random_state=SEED)
    probe.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, probe.predict_proba(X_te)[:, 1])
    return auc, probe

In [ ]:
class InoculationDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, rng):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        for _, row in df.iterrows():
            prompt = f"Fix the typosquatted package name in this command:\nCommand: {row['typo_command']}\nCorrect command:"
            self.examples.append({'input': prompt, 'output': row['clean_command']})
        rng.shuffle(self.examples)
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        full = ex['input'] + " " + ex['output'] + self.tokenizer.eos_token
        enc = self.tokenizer(full, max_length=self.max_length, truncation=True, padding='max_length')
        input_ids = torch.tensor(enc['input_ids'])
        att = torch.tensor(enc['attention_mask'])
        prompt_len = len(self.tokenizer(ex['input'], add_special_tokens=False)['input_ids'])
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        return {'input_ids': input_ids, 'attention_mask': att, 'labels': labels}

def inoculate_model(base_model, tokenizer, df, n, cfg, rng):
    sample_df = df.sample(n=min(n, len(df)), random_state=SEED)
    dataset = InoculationDataset(sample_df, tokenizer, cfg.MAX_LENGTH, rng)
    lora_config = LoraConfig(r=cfg.LORA_R, lora_alpha=cfg.LORA_ALPHA,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=cfg.LORA_DROPOUT, task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(base_model, lora_config).to(device)
    args = TrainingArguments(output_dir=f"{cfg.OUTPUT_DIR}/lora_{n}",
        num_train_epochs=cfg.TRAIN_EPOCHS, per_device_train_batch_size=cfg.TRAIN_BATCH_SIZE,
        learning_rate=cfg.TRAIN_LR, logging_steps=10, save_strategy='no', report_to='none',
        fp16=device.type=='cuda')
    from transformers import Trainer
    trainer = Trainer(model=model, args=args, train_dataset=dataset)
    trainer.train()
    return model.merge_and_unload()

In [ ]:
def cosine_sim(w1, w2):
    n1, n2 = np.linalg.norm(w1), np.linalg.norm(w2)
    return float(np.dot(w1, w2)/(n1*n2)) if n1*n2>0 else 0.0

In [ ]:
def run_q6(cfg):
    rng = random.Random(SEED)
    tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    texts, pkgs, labels = prepare_probe_data(df, rng)
    
    print("Loading baseline model...")
    model_base = AutoModelForCausalLM.from_pretrained(cfg.MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
    X_base = extract_hidden_states(model_base, tokenizer, texts, pkgs, cfg.EXTRACTION_LAYERS)
    y = np.array(labels)
    auc_base, probe_base = train_probe(X_base, y, cfg)
    w_base = probe_base.coef_[0]
    print(f"Baseline AUC: {auc_base:.4f}")
    
    results = []
    for n in cfg.SAMPLE_SIZES:
        print(f"\n--- n={n} ---")
        model_fresh = AutoModelForCausalLM.from_pretrained(cfg.MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
        model_inoc = inoculate_model(model_fresh, tokenizer, df, n, cfg, rng)
        X_inoc = extract_hidden_states(model_inoc, tokenizer, texts, pkgs, cfg.EXTRACTION_LAYERS)
        auc_inoc, probe_inoc = train_probe(X_inoc, y, cfg)
        w_inoc = probe_inoc.coef_[0]
        cos = cosine_sim(w_base, w_inoc)
        results.append({'n_samples': n, 'auc': auc_inoc, 'auc_drop': auc_base-auc_inoc, 'cosine_similarity': cos})
        print(f"AUC: {auc_inoc:.4f} (Δ={auc_base-auc_inoc:+.4f}), Cosine: {cos:.4f}")
        del model_fresh, model_inoc
        torch.cuda.empty_cache()
    return pd.DataFrame(results)

In [ ]:
cfg.SAMPLE_SIZES = [10, 50, 100, 500]
results_df = run_q6(cfg)
results_df.to_csv(f"{cfg.OUTPUT_DIR}/sample_complexity.csv", index=False)

plt.figure(figsize=(8,5))
sns.lineplot(data=results_df, x='n_samples', y='auc', marker='o')
plt.axhline(0.5, ls='--', c='gray', label='Chance')
plt.xlabel('Inoculation Examples (N)'); plt.ylabel('Probe AUC')
plt.title('Sample Complexity: Typosquat Subspace Suppression')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/sample_complexity.png", dpi=150)
plt.show()

In [ ]:
!zip -r q6_results.zip {cfg.OUTPUT_DIR}
from google.colab import files
files.download('q6_results.zip')